In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.gaussian_process import GaussianProcessRegressor
from scipy.stats import uniform, loguniform, norm
from sklearn.gaussian_process.kernels import Matern, RBF, WhiteKernel, ConstantKernel as C
from scipy.optimize import minimize
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score

## **Function 3** - Drug Discovery Project

- **Objective** - drug discovery project, testing combinations of three compounds to create a new medicine.

- **Final Goal** - minimise side effects; in this competition, it is framed as maximisation by optimising a transformed output (e.g. the negative of side effects).

- **Optimization Goal** - maximisation

In [2]:
# Load current cumulative dataset (original + prior weekly updates) for Function 3
X = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_3/initial_inputs.npy')
Y = np.load('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_3/initial_outputs.npy')

# New data for Week 6 (Function 3)
X_w5_new_point = np.array([0.909286, 0.974205, 0.475969], dtype=np.float64)
Y_w5_new_point = np.array([-0.040650730973747384], dtype=np.float64)

# Append the new data point
X_updated = np.vstack((X, X_w5_new_point.reshape(1, -1)))

# Remove duplicate X rows (if point already exists)
X_unique, unique_indices = np.unique(X_updated, axis=0, return_index=True)

# Build Y and keep matching rows only
Y_all = np.append(Y, Y_w5_new_point)
Y_updated = Y_all[unique_indices]

# Save updated arrays
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_3/initial_inputs.npy', X_unique)
np.save('/Users/prathamyeole/Library/CloudStorage/OneDrive-Personal/Imperial - Machine Learning and Artifical Intelligence Certification/Capstone/Weekly Capstone Updates and Comments/Week 1/Initial_data_points_starter/initial_data/function_3/initial_outputs.npy', Y_updated)

print("Updated Inputs (X) - Function 3: ", X_unique)
print("Updated Outputs (Y) - Function 3: ", Y_updated)

Updated Inputs (X) - Function 3:  [[0.04680895 0.23136024 0.77061759]
 [0.13462167 0.21991724 0.45820622]
 [0.15183663 0.43999062 0.99088187]
 [0.17047699 0.6970324  0.14916943]
 [0.17152521 0.34391687 0.2487372 ]
 [0.22054934 0.29782524 0.34355534]
 [0.24211446 0.64407427 0.27243281]
 [0.342299   0.341527   0.47123   ]
 [0.34552327 0.94135983 0.26936348]
 [0.49258141 0.61159319 0.34017639]
 [0.53490572 0.39850092 0.17338873]
 [0.60009728 0.72513573 0.06608864]
 [0.64550284 0.39714294 0.91977134]
 [0.66601366 0.67198515 0.2462953 ]
 [0.74691195 0.28419631 0.22629985]
 [0.90693    0.142503   0.504446  ]
 [0.909286   0.974205   0.475969  ]
 [0.96599485 0.86111969 0.56682913]
 [0.990267   0.061509   0.800335  ]]
Updated Outputs (Y) - Function 3:  [-0.11804826 -0.04800758 -0.39892551 -0.09418956 -0.1121222  -0.04694741
 -0.08796286 -0.01352557 -0.11062091 -0.03483531 -0.11141465 -0.03637783
 -0.11386851 -0.10596504 -0.13146061 -0.04432434 -0.04065073 -0.05675837
 -0.13560967]


### **Output Analysis**
- Last week (W4) we saw $-0.1356$, and this week (W5) I saw $-0.0406$.

- **Real-World Implication**: This is a strong positive move. 
    - In this drug discovery context (where I am likely minimizing side effects or maximizing a efficacy-to-toxicity ratio), I have significantly improved our result.
    - I have successfully moved toward a more favorable combination of compounds.

- **Model Clues**: The jump from -0.13 to -0.04 indicates I am climbing a meaningful slope in this 3D space. 
    - The "chemical landscape" here seems to have a clear directionality. 
    - Since I am still in negative territory, the goal is to see if I can push this value above zero or at least closer to it, suggesting an even more optimal dosage balance.


### **Bayesian Optimisation** Gaussian Process Regressor

- **Model Understanding**: I utilize a Matern kernel with nu=2.5. 
    - This is effective for 3D drug discovery because chemical interactions often exhibit non-linear but continuous behavior, which this kernel captures better than a standard RBF.

- **Changes/Reasons**: I am keeping the model consistent with the Week 5 structure. The current GP successfully identified a path toward a better output (-0.04), so the length-scale parameters are likely starting to converge on a useful representation of the search space.

- **Intended Impact**: To reinforce the surrogate’s confidence in the current high-performing region and allow it to narrow down the most effective combination of the three compounds.


In [3]:
# Define the model - Sticking to the Matern kernel for 3D compound interactions
kernel = Matern(
    length_scale=1.0,
    length_scale_bounds=(1e-2, 1e2), 
    nu=2.5) + \
         WhiteKernel(noise_level=1e-5)

model = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=10)

model.fit(X_unique, Y_updated)

,"kernel kernel: kernel instance, default=NoneThe kernel specifying the covariance function of the GP. If None ispassed, the kernel ``ConstantKernel(1.0, constant_value_bounds=""fixed"")* RBF(1.0, length_scale_bounds=""fixed"")`` is used as default. Note thatthe kernel hyperparameters are optimized during fitting unless thebounds are marked as ""fixed"".",Matern(length...e_level=1e-05)
,"alpha alpha: float or ndarray of shape (n_samples,), default=1e-10Value added to the diagonal of the kernel matrix during fitting.This can prevent a potential numerical issue during fitting, byensuring that the calculated values form a positive definite matrix.It can also be interpreted as the variance of additional Gaussianmeasurement noise on the training observations. Note that this isdifferent from using a `WhiteKernel`. If an array is passed, it musthave the same number of entries as the data used for fitting and isused as datapoint-dependent noise level. Allowing to specify thenoise level directly as a parameter is mainly for convenience andfor consistency with :class:`~sklearn.linear_model.Ridge`.For an example illustrating how the alpha parameter controlsthe noise variance in Gaussian Process Regression, see:ref:`sphx_glr_auto_examples_gaussian_process_plot_gpr_noisy_targets.py`.",1e-10
,"optimizer optimizer: ""fmin_l_bfgs_b"", callable or None, default=""fmin_l_bfgs_b""Can either be one of the internally supported optimizers for optimizingthe kernel's parameters, specified by a string, or an externallydefined optimizer passed as a callable. If a callable is passed, itmust have the signature:: def optimizer(obj_func, initial_theta, bounds): # * 'obj_func': the objective function to be minimized, which # takes the hyperparameters theta as a parameter and an # optional flag eval_gradient, which determines if the # gradient is returned additionally to the function value # * 'initial_theta': the initial value for theta, which can be # used by local optimizers # * 'bounds': the bounds on the values of theta .... # Returned are the best found hyperparameters theta and # the corresponding value of the target function. return theta_opt, func_minPer default, the L-BFGS-B algorithm from `scipy.optimize.minimize`is used. If None is passed, the kernel's parameters are kept fixed.Available internal optimizers are: `{'fmin_l_bfgs_b'}`.",'fmin_l_bfgs_b'
,"n_restarts_optimizer n_restarts_optimizer: int, default=0The number of restarts of the optimizer for finding the kernel'sparameters which maximize the log-marginal likelihood. The first runof the optimizer is performed from the kernel's initial parameters,the remaining ones (if any) from thetas sampled log-uniform randomlyfrom the space of allowed theta-values. If greater than 0, all boundsmust be finite. Note that `n_restarts_optimizer == 0` implies that onerun is performed.",10
,"normalize_y normalize_y: bool, default=FalseWhether or not to normalize the target values `y` by removing the meanand scaling to unit-variance. This is recommended for cases wherezero-mean, unit-variance priors are used. Note that, in thisimplementation, the normalisation is reversed before the GP predictionsare reported... versionchanged:: 0.23",False
,"copy_X_train copy_X_train: bool, default=TrueIf True, a persistent copy of the training data is stored in theobject. Otherwise, just a reference to the training data is stored,which might cause predictions to change if the data is modifiedexternally.",True
,"n_targets n_targets: int, default=NoneThe number of dimensions of the target values. Used to decide the numberof outputs when sampling from the prior distributions (i.e. calling:meth:`sample_y` before :meth:`fit`). This parameter is ignored once:meth:`fit` has been called... versionadded:: 1.3",None
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation used to initialize the centers.Pass an int for reproducible results across multiple function calls.See :term:`

### **Acquisition Function**

- **Function Understanding**: I am using Expected Improvement (EI) for Function 3. 
    - This is a strategic choice for 3D spaces as it specifically targets points that have the highest mathematical probability of exceeding our current best value.

- **Changes/Reasons**: I am maintaining the EI strategy with xi=0.01 and the variable name improvement. 
    - There is no reason to change a winning strategy; the previous query led to a better result, so I should allow EI to continue exploring the frontier of the current maximum.

- **Intended Impact**: To find the optimal balance between exploiting the current -0.04 region and exploring nearby areas that might harbor a true peak (values closer to 0).

In [4]:
def expected_improvement(X, model, y_max, xi=0.01):
    mu, sigma = model.predict(X, return_std=True)
    mu, sigma = mu.reshape(-1, 1), sigma.reshape(-1, 1)
    
    with np.errstate(divide='ignore'):
        # Using improvement as requested
        improvement = mu - y_max - xi
        Z = improvement / sigma
        ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
        ei[sigma <= 0] = 0.0
    return ei.ravel()

# Identify the current best result as the benchmark
y_max = np.max(Y_updated)

# Generate a 3D grid for the dosage space
x_grid = np.random.uniform(0, 1, (20000, 3))

# Calculate EI values
ei_values = expected_improvement(x_grid, model, y_max)

# Select next query
next_query = x_grid[np.argmax(ei_values)]

print(f"Strategic Week 6 Query (Function 3): [{next_query[0]:.6f}-{next_query[1]:.6f}-{next_query[2]:.6f}]")

Strategic Week 6 Query (Function 3): [0.996998-0.991006-0.000966]
